## Model Selection

Before we move on we would like to choose the best possible model for each of the two cases: daily forecast and hourly forecast. It seems that for the daily forecast in the long term you choose linear_order2 in the short to medium term potentially hybrid_order2. For the hourly forecast it seems that just using XGBoost is the best possible model.

The main question is that in the daily forecast if we were to optimise the hyperparameters of the XGBoost model within the hybrid model would it make hybrid_order2 better than linear_order2 in both the short and long term. It would also be nice to do hyperparamter optimisation for the hourly forecast as well just to see whether we can improve the predictions or not. 

Finally it would be good to look at SHAP values to see if we can drop any of the features, particualarly some of the lags as they can make the models computationally expensive.

For our own use (maybe delete later): http://kaggle.com/code/prashant111/a-guide-on-xgboost-hyperparameters-tuning

https://hyperopt.github.io/hyperopt/?source=post_page

https://github.com/hyperopt/hyperopt/wiki/FMin

In [1]:
from hyperopt import hp, fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope
import xgboost as xgb
import pickle
import pandas as pd
from jfk_taxis import load_design, load_models, load_lags, run_forecasts, preprocess, forecast, create_val_data, wrapped_objective, save_hyperparams, load_hyperparams
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import time
import numpy as np
import cupy as cp

In [2]:
# First reload the significant lags
daily_lags = load_lags("daily", "eda")

hourly_lags = load_lags("hourly", "eda")

In [3]:
# Get both the full daily and hourly time series
dir_path = "../data/processed/"
df_daily = pd.read_csv(f"{dir_path}ts_daily2011-2025.csv")
df_hourly = pd.read_csv(f"{dir_path}ts_hour2011-2025.csv")

# Convert dates to datetime objects
df_daily["pickup_date"] = pd.to_datetime(df_daily["pickup_date"])
df_hourly["dt"] = pd.to_datetime(df_hourly["dt"])


In [4]:
# To pass the time series through our helper functions they need to be a pandas series indexed by a datetime object:
ts_hourly = df_hourly["trips"]
ts_hourly.index = df_hourly["dt"]

ts_daily = df_daily["trips"]
ts_daily.index = df_daily["pickup_date"]

In [5]:
# We now need to split into test and train data, we will train on the pre 2024 data and test on 2024 onwards, approx a 90:10 split
ts_daily_train = ts_daily[:"2023-12-31"]
ts_daily_test = ts_daily["2024-01-01":]

ts_hourly_train = ts_hourly[:"2023-12-31"]
ts_hourly_test = ts_hourly["2024-01-01":]

In [6]:
# Define search space
space = {
    # We rely on early stopping when fitting so this isn't an optimised value
    # number of trees
    "n_estimators": 500,

    # Learning rate
    # step size shrinkage, smaller = slower but more precise learning
    "learning_rate": 0.05,

    # Depth/complexity
    # Max depth of tree, larger more complex trees but can cause overfitting
    "max_depth": scope.int(hp.quniform("max_depth", 3, 6, 1)), # scope.int ensures we take ints only
    # minimum "weight" needed in child node. Higher values more conservative, fewer splits helps prevent overfitting
    "min_child_weight": hp.loguniform("min_child_weight", -2.3, 2.3), # approx [0.1, 10]

    # Randomisation/feature subsampling
    # fraction of rows used per tree, lower adds randomness reduces overfitting
    "subsample": hp.uniform("subsample", 0.6, 1.0),
    # fraction of features used per tree
    "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),

    # Regularisation
    # L2 penalty, good range is [0.1, 10] we use loguniform because this means that every order of magnitude has equal probability, the def of log uniform in hyperopt is that it returns a value exp(U(low,high)) where U is uniform dist.  
    "reg_lambda": hp.loguniform("reg_lambda", np.log(1e-2), np.log(100)), # [0.01, 100]
    # L1 penalty
    "reg_alpha": hp.loguniform("reg_alpha", np.log(1e-3), np.log(10)), # [0.001, 10]

    # Split pnealty (gamma) 
    # minimum loss reduction required to split a node, higher values = more conservative
    "gamma": hp.loguniform("gamma", -7.0, 2.3), # approx [0.0009, 10]

    "random_state": 37,
    #"early_stopping_rounds": 100,
    "eval_metric": "mae", 
    "tree_method": "hist",
    "device": "cuda" # Use GPU if available
}
    
    

There are a few interesting things we would like to vary when doing our Bayesian hyperparamter optimisation. The first is that we want to both include and exclude COVID from the data we use to tune hyperparamters on.

The reason for this is because during COVID the usual seasonality breaks down dramatically as we get unusal travel patterns. So it may be worth tuning a model on data without COVID as then it is being asseseed more on its ability to pick up the more "normal" patterns within the data. You then have a question of do you use pre or post COVID data, there is more pre COVID data but it will obviously be less relevant for forecasting in the present. Alternatively it may be actually be worth including the COVID data as then the model is tuned to be more robust to "unusal" regimines within the data.

So as it's not clear which of the three will be optimal we will just run all three and then compare the models that hyperopt finds.

For the daily series we wil use a 30 day forecast in our objection function to validate with. For the hourly series we will do a weekly forecast (168 hours).

In [ ]:
# Dictionary of parameters for Baysian optimisation
bayes_dict = {}

In [ ]:
# Daily non_linear pre, incl and post COVID

# Set the parameters for creat_val_data and the objective function
n_splits = 5
test_size = 30
lags = daily_lags
constant = False
order = 0
fourier_features = ["YE", "W"]
time_step = "D"
hybrid = None
steps = 30


# Looping through this dict avoids having to have three separate code cells, the keys are the sigs, values are the tsk
tmp_dict = {
    "daily_non_linear_pre_COVID": ts_daily_train[:"2020-01-01"],
    "daily_non_linear_incl_COVID": ts_daily_train,
    "daily_non_linear_post_COVID": ts_daily_train["2022-01-01":]
}

# Add to Bayes dict
for key, value in tmp_dict.items():
    bayes_dict[key] = {
        "n_splits" : n_splits,
        "test_size" : test_size,
        "lags" : lags,
        "constant" : constant,
        "order" : order,
        "fourier_features" : fourier_features,
        "time_step" : time_step,
        "ts" : value,
        "hybrid" : hybrid,
        "steps" : steps
    }

In [ ]:
# Hourly non_linear pre, incl and post COVID

# Set the parameters for creat_val_data and the objective function
n_splits = 5
test_size = 168
lags = hourly_lags
constant = False
order = 0
fourier_features = ["D", "W"]
time_step = "h"
hybrid = None
steps = 168


# Looping through this dict avoids having to have three separate code cells, the keys are the sigs, values are the tsk
tmp_dict = {
    "hourly_non_linear_pre_COVID": ts_hourly_train[:"2020-01-01"],
    "hourly_non_linear_incl_COVID": ts_hourly_train,
    "hourly_non_linear_post_COVID": ts_hourly_train["2022-01-01":]
}

# Add to Bayes dict
for key, value in tmp_dict.items():
    bayes_dict[key] = {
        "n_splits" : n_splits,
        "test_size" : test_size,
        "lags" : lags,
        "constant" : constant,
        "order" : order,
        "fourier_features" : fourier_features,
        "time_step" : time_step,
        "ts" : value,
        "hybrid" : hybrid,
        "steps" : steps
    }


In [ ]:
# Daily non_linear incl COVID

# Set the parameters for creat_val_data and the objective function
n_splits = 5
test_size = 30
lags = daily_lags
constant = False
order = 0
fourier_features = ["YE", "W"]
time_step = "D"
ts = ts_daily_train
hybrid = None
steps = 30
sig = "daily_non_linear_incl_COVID"


bayes_dict[sig] = {
    "n_splits" : n_splits,
    "test_size" : test_size,
    "lags" : lags,
    "constant" : constant,
    "order" : order,
    "fourier_features" : fourier_features,
    "time_step" : time_step,
    "ts" : ts,
    "hybrid" : hybrid,
    "steps" : steps
}

In [ ]:
# Daily non_linear post COVID

# Set the parameters for creat_val_data and the objective function
n_splits = 5
test_size = 30
lags = daily_lags
constant = False
order = 0
fourier_features = ["YE", "W"]
time_step = "D"
ts = ts_daily_train["2022-01-01":]
hybrid = None
steps = 30
sig = "daily_non_linear_post_COVID"


bayes_dict[sig] = {
    "n_splits" : n_splits,
    "test_size" : test_size,
    "lags" : lags,
    "constant" : constant,
    "order" : order,
    "fourier_features" : fourier_features,
    "time_step" : time_step,
    "ts" : ts,
    "hybrid" : hybrid,
    "steps" : steps
}

In [ ]:
# Hourly non_linear pre COVID

# Set the parameters for creat_val_data and the objective function
n_splits = 5
test_size = 168
lags = hourly_lags[:168]
constant = False
order = 0
fourier_features = ["D", "W"]
time_step = "h"
ts = ts_hourly_train
steps = 168
hybrid = None
sig = "hourly_non_linear_pre_COVID"

bayes_dict[sig] = {
    "n_splits" : n_splits,
    "test_size" : test_size,
    "lags" : lags,
    "constant" : constant,
    "order" : order,
    "fourier_features" : fourier_features,
    "time_step" : time_step,
    "ts" : ts,
    "hybrid" : hybrid,
    "steps" : steps
}

In [8]:
# Create the folds
fold_dict = create_val_data(n_splits, test_size, lags, constant, order, fourier_features, time_step, ts)

Fold 0
[   0    1    2 ... 3135 3136 3137]
Fold 1
[   0    1    2 ... 3165 3166 3167]
Fold 2
[   0    1    2 ... 3195 3196 3197]
Fold 3
[   0    1    2 ... 3225 3226 3227]
Fold 4
[   0    1    2 ... 3255 3256 3257]


In [9]:
# Set attributes of wrapped_objective
wrapped_objective.fold_dict = fold_dict
wrapped_objective.lags = lags
wrapped_objective.steps = steps
wrapped_objective.hybrid = hybrid


In [ ]:
# Optimisation algorithm
trials = Trials()

best_hyperparams = fmin(fn = wrapped_objective,
                        space = space,
                        algo = tpe.suggest,
                        max_evals = 100,
                        trials = trials)

(2767, 360)                                            
Fit time: 1.92 seconds                                 
Predict time: 0.3359 seconds                           
(2797, 360)                                            
Fit time: 1.82 seconds                                 
Predict time: 0.3038 seconds                           
(2827, 360)                                            
Fit time: 1.81 seconds                                 
Predict time: 0.3049 seconds                           
(2857, 360)                                            
Fit time: 1.80 seconds                                 
Predict time: 0.3053 seconds                           
(2887, 360)                                            
Fit time: 1.75 seconds                                 
Predict time: 0.2922 seconds                           
MAEs:                                                  
[412.58811848958334, 428.69365234375, 341.0744140625, 1098.099169921875, 670.9555989583333]
Avg MAE:    

In [18]:
print("The best hyperparamters are: ", "\n")
print(best_hyperparams)

The best hyperparamters are:  

{'colsample_bytree': np.float64(0.7461563082314766), 'gamma': np.float64(0.3330516420563867), 'max_depth': np.float64(5.0), 'min_child_weight': np.float64(0.979909668812688), 'reg_alpha': np.float64(9.084977135335684), 'reg_lambda': np.float64(4.611255387226133), 'subsample': np.float64(0.8402110494723659)}


In [ ]:
# Save hyperparams
save_hyperparams(best_hyperparams, sig)

In [ ]:
# Load hyperparameters
sig = "daily_non_linear"

best_hyperparams = load_hyperparams(sig)

In [19]:
# It would now be interesting to use this hyperparams and use them on the forecasts from the previous notebook to see how they compare

# Load the previous non linear model
linear_models_loaded, non_linear_models_loaded = load_models("5_order_linear_daily")

# Load the previous non linear design matrix
linear_design_loaded, non_linear_design_loaded = load_design("5_order_linear_daily")

# Get design, target and dp
X = non_linear_design_loaded["base_non_linear"][0]
y = non_linear_design_loaded["base_non_linear"][1]
dp = non_linear_design_loaded["base_non_linear"][2]

# Now create our new non linear model trained on the full training data set
new_non_linear = xgb.XGBRegressor(
        n_estimators = space["n_estimators"],
        learning_rate = space["learning_rate"],
        max_depth = best_hyperparams["max_depth"],
        min_child_weight = best_hyperparams["min_child_weight"],
        subsample = best_hyperparams["subsample"],
        colsample_bytree = best_hyperparams["colsample_bytree"],
        gamma = best_hyperparams["gamma"],
        reg_alpha = best_hyperparams["reg_alpha"],
        reg_lambda = best_hyperparams["reg_lambda"],
        random_state = 37,
        eval_metric = "mae",
        tree_method = "hist",
        device = "cuda"
        )


new_non_linear.fit(X, y,
    verbose = False)

non_linear_models_loaded["new_non_linear"] = (new_non_linear, dp, None)

FileNotFoundError: [Errno 2] No such file or directory: 'T:\\Coding\\Summer_2025\\NYC-taxi-data-analysis\\data\\saved_objects/Linear_keys_5_order_linear_daily_models.pkl'

In [ ]:
# Steps for the forecast
steps = [1, 2, 3, 7, 14, 28, 30, 60, 180, 365, 500, 546]


In [ ]:
# Run forecasts
run_forecasts(steps, daily_lags, {}, non_linear_models_loaded, False, "D", ts_daily_train, ts_daily_test)